# Zero-Train Optimization of a Healthcare Chatbot
Medical Chatbot:

Background: Medical advice chatbot (LLM) classifier model trained on based on Llama3.2 on online medical forums.


Dataset: https://huggingface.co/datasets/lextale/FirstAidInstructionsDataset/viewer/default/icliniqDataset


Scenario: Optimize the model without additional training.


Use Case: The model is already trained and we want to optimize it without additional training.

## Setup

### Install dependencies and utility functions: This cell should be run once.

In [1]:
%%capture

%pip install matplotlib numpy pandas requests tqdm ipywidgets
%pip install --ignore-installed 'git+https://github.com/Authentrics-ai/authentrics-client.git@v2.4.0'

from pathlib import Path

import os
import authentrics_client as authrx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import random
from datetime import datetime
import time
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from threading import Thread
import contextlib

# Utility functions for loading indicators
@contextlib.contextmanager
def loading_spinner(message="Processing..."):
    """Context manager for showing a loading spinner"""
    spinner_widget = widgets.HTML(value=f"""
    <div style="display: flex; align-items: center; gap: 10px;">
        <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
        <span style="font-size: 14px; color: #555;">{message}</span>
    </div>
    <style>
    @keyframes spin {{
        0% {{ transform: rotate(0deg); }}
        100% {{ transform: rotate(360deg); }}
    }}
    </style>
    """)
    display(spinner_widget)
    try:
        yield
    finally:
        spinner_widget.close()

def show_progress_bar(total, description="Progress"):
    """Create and return a progress bar"""
    return tqdm(total=total, desc=description, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

def wait_for_analytics(project_id, min_wait_time=30, max_wait_time=300, check_interval=10):
    """
    Wait for ALL analytics to be complete before continuing.
    This prevents NaN values from appearing in the summary table.
    """
    import time

    print("🔄 Waiting for analytics to process...")
    countdown_widget = widgets.HTML()
    display(countdown_widget)

    # Minimum wait with countdown
    for remaining in range(min_wait_time, 0, -1):
        countdown_widget.value = f"""
        <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
            <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
            <span style="font-size: 14px; color: #555;">Minimum wait period: {remaining} seconds remaining...</span>
        </div>
        <style>
        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}
        </style>
        """
        time.sleep(1)

    # Poll until ALL analytics are complete
    start_polling = time.time()
    attempt = 1

    while (time.time() - start_polling) < (max_wait_time - min_wait_time):
        try:
            project = client.project.get_project_by_id(project_id)
            files = project["fileList"][1:]  # Skip base model

            # Check that ALL files have both weight and bias contributions
            files_missing_analytics = []
            for f in files:
                if (f.get("totalWeightContribution") is None or
                    f.get("totalBiasContribution") is None):
                    files_missing_analytics.append(f["fileName"])

            # Only continue if ALL analytics are complete
            if not files_missing_analytics:
                countdown_widget.value = """
                <div style="color: green; font-weight: bold; margin: 10px 0;">
                    ✅ All analytics are complete!
                </div>
                """
                return True

            # Show progress with specific missing files
            total_files = len(files)
            completed_files = total_files - len(files_missing_analytics)
            progress_pct = (completed_files / total_files * 100) if total_files > 0 else 0

            missing_display = ', '.join(files_missing_analytics[:3])
            if len(files_missing_analytics) > 3:
                missing_display += f" and {len(files_missing_analytics) - 3} more"

            countdown_widget.value = f"""
            <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
                <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #orange; border-radius: 50%; animation: spin 1s linear infinite;"></div>
                <span style="font-size: 14px; color: #555;">
                    Attempt {attempt}: {completed_files}/{total_files} files complete ({progress_pct:.1f}%)
                    <br/>⏳ Waiting for: {missing_display}
                </span>
            </div>
            <style>
            @keyframes spin {{
                0% {{ transform: rotate(0deg); }}
                100% {{ transform: rotate(360deg); }}
            }}
            </style>
            """
            attempt += 1
            time.sleep(check_interval)

        except Exception as e:
            print(f"⚠️ Error checking analytics status: {e}")
            time.sleep(check_interval)

    # Timeout reached - show final status
    try:
        project = client.project.get_project_by_id(project_id)
        files = project["fileList"][1:]
        files_missing_analytics = [f["fileName"] for f in files
                                 if (f.get("totalWeightContribution") is None or
                                     f.get("totalBiasContribution") is None)]

        if files_missing_analytics:
            countdown_widget.value = f"""
            <div style="color: red; font-weight: bold; margin: 10px 0;">
                ⚠️ Timeout: Analytics incomplete for {len(files_missing_analytics)} files.
                <br/>NaN values will appear for: {', '.join(files_missing_analytics[:5])}
                <br/>Consider increasing wait time or contacting support.
            </div>
            """
            return False
        else:
            countdown_widget.value = """
            <div style="color: green; font-weight: bold; margin: 10px 0;">
                ✅ All analytics completed just in time!
            </div>
            """
            return True
    except Exception as e:
        countdown_widget.value = f"""
        <div style="color: red; font-weight: bold; margin: 10px 0;">
            ❌ Error checking final status: {e}
        </div>
        """
        return False

def inference(
    client: authrx.AuthentricsClient,
    optimized_model_path: str,
    prompt: list[str] | Path,
    batch_size: int = 1,
):
    base_model_path = "demo-models/Medical-Chatbot/iteration_24.tar"
    
    # Single stimulus
    if isinstance(prompt, Path):
        optimized_result = client.dynamic.direct_inference(
            model_path=optimized_model_path,
            stimulus_path=prompt,
            format="HF_TEXT_GENERATION",
            inference_config={"max_new_tokens": 50},
        )
        base_result = client.dynamic.direct_inference(
            model_path=base_model_path,
            stimulus_path=prompt,
            format="HF_TEXT_GENERATION",
            inference_config={"max_new_tokens": 50},
        )
        return {
            "prompt": prompt,
            "model_responses": [
                base_result["model_responses"][0],
                optimized_result["model_responses"][0],
            ],
        }

    # Multiple stimuli
    else:
        optimized_response = client.dynamic.batch_direct_inference(
            model_path=optimized_model_path,
            stimulus_paths=prompt,
            format="HF_TEXT_GENERATION",
            batch_size=batch_size,
            inference_config={"max_new_tokens": 50},
        )
        base_response = client.dynamic.batch_direct_inference(
            model_path=base_model_path,
            stimulus_paths=prompt,
            format="HF_TEXT_GENERATION",
            batch_size=batch_size,
            inference_config={"max_new_tokens": 50},
        )
        
        return {
            "prompt": prompt,
            "model_responses": [
                base_response["model_responses"][0],
                optimized_response["model_responses"][0],
            ],
        }


def display_static_analysis_results(result: dict):
    print(f"Weight summary score: {result['weight_summary_score']}")
    pd.options.display.max_rows = None

        
    methods = {"min": np.amin, "mean": np.mean, "std": np.std, "max": np.amax}
    awd = {"name": [], "min": [], "mean": [], "std": [], "max": []}

    for name, values in response["result"]["absolute_weight_difference"].items():
        awd["name"].append(name)
        for method_name, method in methods.items():
            awd[method_name].append(method(values))

    df = pd.DataFrame(awd)
    display(df)


### Contact Authentrics for the URL and user credentials info@authentrics.ai

In [2]:

DEMO_SERVER_URL = input("Enter your Authentrics URL: ")
PROJECT_NAME = "Healthcare Chatbot ZTO"
pd.options.display.precision = 3
pd.options.display.chop_threshold = None

### Establish a session with authentrics
- Rerun this cell if your session expires

In [3]:
client = authrx.AuthentricsClient(DEMO_SERVER_URL)
client.auth.login()
print("✅ Session established successfully!")

✅ Session established successfully!


### Generate a project in Authentrics to track checkpoints, and perform asynchronous analytics
1. Creates a project for checkpoint management.
2. **Option A**: Point to existing checkpoint storage (no data movement required).
3. **Option B**: Upload checkpoints directly through our client (automated storage and versioning).

In [5]:
projects = client.project.get_projects()
if projects is not None:
    project = next((p for p in projects if p["name"].startswith(PROJECT_NAME)), None)
    if project:
        client.project.delete_project(project["id"], hard_delete=True)

PROJECT_NAME = PROJECT_NAME + " " + str(int(datetime.now().timestamp()))
project = client.project.create_project(
    PROJECT_NAME,
    "A smaller LLM specializing in medical advice",
    authrx.FileType.HF_TEXT,
)
project_id = project["id"]

### Restore all checkpoint files, stimulus files, and expected output file in bucket to a stable version

In [6]:
with loading_spinner("Please wait while we restore the requested files..."):
    client.get('/transfer/MedChat')

HTML(value='\n    <div style="display: flex; align-items: center; gap: 10px;">\n        <div style="width: 20p…

### Point authentrics file registry to the checkpoints using the api

In [7]:
print("Adding checkpoints to project...")
with show_progress_bar(5, "Adding checkpoints") as pbar:
    for i in range(20, 25):
        client.checkpoint.add_external_checkpoint(
            project_id,
            f"demo-models/Medical-Chatbot/iteration_{i}.tar",
            authrx.FileType.HF_TEXT,
            file_name=f"iteration_from_dataset_{i}.tar",
            tag=f"v{i}",
        )
        pbar.update(1)

project = client.project.get_project_by_name(PROJECT_NAME)
project_id = project["id"]
print(f"✅ Project created with id: {project['id']}")
print(f"Project name: {project['name']}")
for file in project["fileList"]:
    print(f"File name: {file['fileName']}")

Adding checkpoints to project...


Adding checkpoints:   0%|          | 0/5 [00:00<?]

✅ Project created with id: 6941922cb3366c63a1c6e449
Project name: Healthcare Chatbot ZTO 1765904940
File name: iteration_from_dataset_20.tar
File name: iteration_from_dataset_21.tar
File name: iteration_from_dataset_22.tar
File name: iteration_from_dataset_23.tar
File name: iteration_from_dataset_24.tar


Wait for Analytics to be ready before proceeding.

In [8]:
print("\n" + "="*60)
print("📊 ANALYTICS PROCESSING")
print("="*60)
print("Authentrics is now computing analytics in the background.")
print("This includes weight contributions and bias scores for each checkpoint.")

analytics_ready = wait_for_analytics(project_id, min_wait_time=10, max_wait_time=300)
if not analytics_ready:
    print("\n❌ Analytics are not complete.")
    print("Please wait a moment longer and then continue. Please contact support if the issue persists.")


📊 ANALYTICS PROCESSING
Authentrics is now computing analytics in the background.
This includes weight contributions and bias scores for each checkpoint.
🔄 Waiting for analytics to process...


HTML(value='')

## Zero-Train Optimization (ZTO) Results

Authentrics.ai provides a zero-train optimization (ZTO) service that allows you to optimize a model's performance without additional training.

This will increase or decrease the effects of each training iteration on the model to obtain the best possible performance. In this case, the performance is measured by the semantic similarity between the model's response and the expected output.

The scaling factor limit is the maximum fraction of change that any one training iteration can undergo. In this case, each training iteration can only change by 10% of its original effect.

In [9]:
stimulus_paths = [f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)]
expected_output_path = "demo-models/Medical-Chatbot/stimuli/expected_output.txt"

response = client.dynamic.zero_train_optimizer(
    project_id=project["id"],
    scaling_factor_limit=0.1,
    stimulus_paths=stimulus_paths,
    batch_size=10,
    expected_output_path=expected_output_path,
    inference_config={"max_new_tokens": 100},
)

result = response["result"]

print(result)

{'model_path': '6941922cb3366c63a1c6e449/results/9934c9e2-2757-4359-9bd0-0128e36bc42a.tar', 'trained_model_error': 0.9514996968209743, 'optimized_model_error': 0.9262264221906662, 'optimized_scaling_factors': [-0.07704337546693826, -0.08660429046552424, -0.09988516978916917, -0.034706598966179], 'number_of_inferences': 50}


### Result details

`model_path` is the path to the optimized model in the Authentrics bucket.

In [10]:
result["model_path"]

'6941922cb3366c63a1c6e449/results/9934c9e2-2757-4359-9bd0-0128e36bc42a.tar'

`base_model_error` is the semantic similarity between the trained model's response and the expected output.

In [11]:
result["trained_model_error"]

0.9514996968209743

`best_model_error` is the semantic similarity between the optimized model's response and the expected output.

In [12]:
result["optimized_model_error"]

0.9262264221906662

`optimized_scaling_factors` is the scaling factor for each training iteration that was applied to the model to produce the optimized model.

In [13]:
result["optimized_scaling_factors"]

[-0.07704337546693826,
 -0.08660429046552424,
 -0.09988516978916917,
 -0.034706598966179]

`number_of_inferences` is the number of inferences that were performed to produce the optimized model.

In [14]:
result["number_of_inferences"]

50

### Inference the latest and optimized models

In [15]:
import json

optimized_model_path = result['model_path']
latest_model_path = client.project.get_project_by_id(project_id)['fileList'][-1]['filePath']

prompt_path = Path("prompt_00.json")
latest_response = client.dynamic.direct_inference(
    model_path=latest_model_path,
    stimulus_path=prompt_path,
    format="HF_TEXT_GENERATION",
    inference_config={"max_new_tokens": 100},
)
optimized_response = client.dynamic.direct_inference(
    model_path=optimized_model_path,
    stimulus_path=prompt_path,
    format="HF_TEXT_GENERATION",
    inference_config={"max_new_tokens": 100},
)
with open(prompt_path, "r") as f:
    prompt = json.load(f)[0]["content"]

print(f"User prompt:\n\n\t{prompt}\n")
print(
    "Latest model response:"
    f"\n\n\t{latest_response['result']["output"][0][0]["generated_text"][1]["content"]}\n"
)
print(
    "Optimized model response:"
    f"\n\n\t{optimized_response['result']["output"][0][0]["generated_text"][1]["content"]}\n"
)

User prompt:

	Hello sir,  My son has sinusitis and also nasal polyps, we got to know this recently. He suffers from a nose block and breathing issues. So when we visited a PCP (primary care physician), he suggested using nasal spray. So my doubt is, how can nasal spray efficiently relieve symptoms of sinusitis and nasal polyps, and what mechanisms make it useful for people suffering from these conditions? Could you also elaborate on how they work to reduce inflammation and congestion, and any possible negative effects or things to watch out for when using nasal sprays to treat sinusitis and nasal polyps?   .

Latest model response:

	I can understand your concern.  Nasal spray is a common treatment for sinusitis and nasal polyps. Nasal sprays work by reducing inflammation and congestion in the nasal passages. They typically contain ingredients such as Steroid (fluticasone propionate), Decongestants (Salbutamol), or Antihistamines (Azelastine). Steroids work by reducing inflammation in

## Examples of error scores used to optimize the model

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
def compute_error_score(ss_model: SentenceTransformer, model_outputs: list[str], expected_outputs: list[str]):
    if len(model_outputs) != len(expected_outputs):
        raise ValueError("model_output and expected_output must have the same length")

    generated_embeddings = ss_model.encode(model_outputs)
    expected_embeddings = ss_model.encode(expected_outputs)
    
    score = ss_model.similarity_pairwise(
        generated_embeddings,
        expected_embeddings,
    )

    return 1.0 - score

### Load prompts and expected outputs

In [18]:
import json

with open("expected_output.txt", "r") as f:
    expected_outputs = f.readlines()

prompts = []
for i in range(10):
    with open(f"prompt_{i:02d}.json", "r") as f:
        prompts.append(json.load(f)[0]["content"])

### Inference the models to get the model outputs

In [19]:
optimized_response = client.dynamic.batch_direct_inference(
    model_path=optimized_model_path,
    stimulus_paths=[
        f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)
    ],
    format="HF_TEXT_GENERATION",
    batch_size=10,
    inference_config={"max_new_tokens": 100},
)
latest_response = client.dynamic.batch_direct_inference(
    model_path=latest_model_path,
    stimulus_paths=[
        f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)
    ],
    format="HF_TEXT_GENERATION",
    batch_size=10,
    inference_config={"max_new_tokens": 100},
)

latest_model_outputs = [response[0]["generated_text"][1]["content"] for response in latest_response["result"]["output"]]
optimized_model_outputs = [response[0]["generated_text"][1]["content"] for response in optimized_response["result"]["output"]]

### Test with the model we use

In [20]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

latest_model_scores = compute_error_score(model, latest_model_outputs, expected_outputs)
optimized_model_scores = compute_error_score(model, optimized_model_outputs, expected_outputs)

pd.DataFrame({
    "prompt": prompts,
    "latest_model_score": latest_model_scores,
    "optimized_model_score": optimized_model_scores,
})

,prompt,latest_model_score,optimized_model_score
0,"Hello sir, My son has sinusitis and also nasa...",0.470,0.533
1,I read in the news that coronavirus is again s...,0.991,0.978
2,I am a 30-year-old female working as a traffic...,0.909,1.001
3,I have a question about my father. He is 75 ye...,0.875,0.953
4,"My child is 10 years old. As a parent, I want ...",1.011,1.001
5,"The fasting blood sugar of my mother, aged 55 ...",0.922,0.943
6,I had a sudden hearing loss twice and because ...,1.086,1.037
7,I am a 23-year-old female. I would like to kno...,1.074,0.917
8,"I am a 22-year-old female weighing 90 pounds, ...",1.010,0.995
9,Last night I looked directly at a 55 watt germ...,0.888,0.939


### Test with another model


In [21]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")

latest_model_scores = compute_error_score(model, latest_model_outputs, expected_outputs)
optimized_model_scores = compute_error_score(model, optimized_model_outputs, expected_outputs)

pd.DataFrame({
    "prompt": prompts,
    "latest_model_score": latest_model_scores,
    "optimized_model_score": optimized_model_scores,
})

,prompt,latest_model_score,optimized_model_score
0,"Hello sir, My son has sinusitis and also nasa...",0.420,0.461
1,I read in the news that coronavirus is again s...,0.964,0.984
2,I am a 30-year-old female working as a traffic...,1.018,0.943
3,I have a question about my father. He is 75 ye...,0.885,0.965
4,"My child is 10 years old. As a parent, I want ...",0.922,0.978
5,"The fasting blood sugar of my mother, aged 55 ...",0.980,0.918
6,I had a sudden hearing loss twice and because ...,1.061,0.977
7,I am a 23-year-old female. I would like to kno...,1.043,0.849
8,"I am a 22-year-old female weighing 90 pounds, ...",1.056,1.019
9,Last night I looked directly at a 55 watt germ...,0.915,0.938


## Analyze the optimized model

Upload the optimized model to the project


In [22]:
project = client.checkpoint.add_external_checkpoint(
    project_id,
    optimized_model_path.split("/", 3)[-1],
    authrx.FileType.HF_TEXT,
    file_name="optimized_model.tar",
    tag="optimized",
)
checkpoint_id = project["fileList"][-1]["id"]

Wait for analytics to be ready


In [24]:
analytics_ready = wait_for_analytics(project_id, min_wait_time=10, max_wait_time=300)

🔄 Waiting for analytics to process...


HTML(value='')

KeyboardInterrupt: 

Run static analysis on the optimized model

In [25]:
response = client.static.static_analysis(
    project_id=project_id,
    checkpoint_id=checkpoint_id,
)

HTTPError: 500 Server Error: Internal Server Error for url: https://cloudrun-authentrics-gateway-32869338103.us-central1.run.app/static_analysis

In [ ]:
display_static_analysis_results(response["result"])


Weight summary score: 9.827903704717755e-05


,name,min,mean,std,max
0,base_model.model.model.layers.0.mlp.down_proj....,-2.287e-05,-1.201e-08,8.205e-06,2.365e-05
1,base_model.model.model.layers.0.mlp.down_proj....,-4.665e-05,7.118e-07,1.279e-05,4.359e-05
2,base_model.model.model.layers.0.mlp.gate_proj....,-4.924e-05,4.355e-07,1.478e-05,5.188e-05
3,base_model.model.model.layers.0.mlp.gate_proj....,-1.691e-05,1.779e-07,5.860e-06,1.504e-05
4,base_model.model.model.layers.0.mlp.up_proj.lo...,-3.943e-05,9.909e-07,1.312e-05,3.709e-05
5,base_model.model.model.layers.0.mlp.up_proj.lo...,-2.494e-05,-1.539e-07,7.932e-06,2.714e-05
6,base_model.model.model.layers.0.self_attn.k_pr...,-4.561e-05,-4.303e-07,1.604e-05,4.737e-05
7,base_model.model.model.layers.0.self_attn.k_pr...,-5.820e-05,-6.866e-08,1.569e-05,5.972e-05
8,base_model.model.model.layers.0.self_attn.o_pr...,-4.632e-05,-1.014e-06,1.610e-05,4.407e-05
9,base_model.model.model.layers.0.self_attn.o_pr...,-4.253e-05,-1.052e-06,1.363e-05,4.118e-05
